In [1]:
import re

def parse_fountain(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    # Common Fountain transitions and directions to exclude as characters
    NON_CHARACTER_ELEMENTS = {
        "FADE IN", "FADE OUT", "FADE TO BLACK", "FADE TO WHITE",
        "CUT TO", "CUT TO BLACK", "CUT TO WHITE",
        "DISSOLVE TO", "MATCH CUT", "SMASH CUT",
        "INT", "EXT", "INT.", "EXT.",  # Scene heading prefixes (if standalone)
        "CONTINUED", "CONTINUOUS", "END", "TO BLACK", "TO WHITE"
    }
    
    characters = {}  # Dictionary to store character appearances and dialogues
    locations = {}   # Dictionary to store location occurrences and associated scenes
    current_character = None
    current_dialogue = []
    current_location = None
    
    for line in lines:
        line = line.rstrip()  # Remove trailing spaces and newlines
        
        # Scene headings (INT. or EXT.)
        scene_match = re.match(r'^(INT\.\s*|EXT\.\s*)(.+)', line)
        if scene_match:
            location = scene_match.group(2).strip()
            current_location = location
            if location not in locations:
                locations[location] = {"occurrences": 0, "scenes": []}
            locations[location]["occurrences"] += 1
            locations[location]["scenes"].append(line)
        
        # Character names (UPPERCASE, limited words, excluding transitions)
        elif (re.match(r'^[A-Z][A-Z0-9 ]+$', line) and 
              len(line.split()) <= 3 and 
              line not in NON_CHARACTER_ELEMENTS):
            if current_character and current_dialogue:
                characters[current_character]["dialogues"].append(" ".join(current_dialogue))
                current_dialogue = []
            current_character = line
            if current_character not in characters:
                characters[current_character] = {"occurrences": 0, "dialogues": []}
            characters[current_character]["occurrences"] += 1
        
        # Dialogue lines
        elif current_character and line.strip():
            current_dialogue.append(line)
        
        # Blank line (reset character)
        elif line.strip() == "":
            if current_character and current_dialogue:
                characters[current_character]["dialogues"].append(" ".join(current_dialogue))
            current_character = None
            current_dialogue = []
    
    return {"characters": characters, "locations": locations}


In [2]:

# Example usage
fountain_file = "script.fountain"  # Replace with your file path
parsed_data = parse_fountain(fountain_file)

print("Characters:")
for character, details in parsed_data['characters'].items():
    print(f"{character} - Appearances: {details['occurrences']}")
    print("Sample Dialogue:")
    for dialogue in details['dialogues'][:3]:  # Show up to 3 sample lines
        print(f"  {dialogue}")
    print()

print("Locations:")
for location, details in parsed_data['locations'].items():
    print(f"{location} - Occurrences: {details['occurrences']}")
    print("Sample Scenes:")
    for scene in details['scenes'][:3]:  # Show up to 3 sample scene headings
        print(f"  {scene}")
    print()


Characters:
ENFORCER - Appearances: 4
Sample Dialogue:
  Stop! In violation of Creative Suppression Act 2.7!
  All units! Target heading east through server block D!
  Left tunnel! Move in!

PHOENIX - Appearances: 114
Sample Dialogue:
  (whispering to herself)
  (whispers)
  (to herself)

ELDERLY ARTIST - Appearances: 5
Sample Dialogue:
  (gesturing to a floating artwork)
  (nodding toward Phoenix)
  When we create together, our art becomes more than marks on a wall. It becomes a voice - their worst nightmare.

KAI - Appearances: 3
Sample Dialogue:
  And now the stories are spreading. My cousin saw one of your murals, Phoenix. Said it gave her hope for the first time in years.
  She's right. Your art... it makes people stop. Talk to each other. Share stories.
  My whole block has been talking about your latest piece. The one with the broken chains?

MAYA - Appearances: 4
Sample Dialogue:
  That's what they want - to isolate us. But look around. Every piece of art here tells the same st